# Online RL Thesis — Sprint 0 addendum · P0 Profiling

**目的**：在 §2 sensor envelope (18 run × 600k 步) 启动前，确认两件事：

| 子任务 | 问题 | 决策 |
|---|---|---|
| **P0a** | wallclock 1.5h/run 时间花在哪？是 NN 还是 CPU env？ | 决定是否值得做 Numba JIT |
| **P0b** | `num_envs=12`（吃满 L4 Colab 的 12 个 hyperthread）是否能近似翻倍？ | 决定 thesis matrix 的 `num_envs` 协议（**最后窗口期**：thesis 启动后该协议固定不可变） |

**前置**：[`sac_thesis_s0_preflight.ipynb`](sac_thesis_s0_preflight.ipynb) 的 P1 + P2 已完成。

**预算**：~15 分钟 wallclock（cProfile 2 min + 两次 50k 各 ~5-7 min）。


## 0. 环境检查


In [1]:
!nvidia-smi


Sat Apr 25 18:04:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 物理核与逻辑核数（决定 num_envs 上限）
import os, multiprocessing as mp
print(f"CPU count:      {mp.cpu_count()} (logical threads)")
print(f"sched affinity: {len(os.sched_getaffinity(0))}")


PyTorch:        2.10.0+cu128
CUDA available: True
CPU count:      12 (logical threads)
sched affinity: 12


## 1. 挂载 Drive + cd


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd


/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. P0a — cProfile：定位瓶颈

跑 10k 步（含 5k random + 5k update），保存 prof 文件，打印 cumulative time top 30。

预期：如果 `vehicle.py` / `flow.py` / `env.py` 加起来 ≥ 50% → CPU env stepping 是瓶颈，可考虑 Numba JIT 这条线（Tier 3）。如果 PyTorch 相关 ≥ 50% → GPU pipeline 出问题，需要排查（不太可能）。


In [ ]:
import os, subprocess, time
from pathlib import Path

PROF_PATH = Path('/tmp/sac_p0a.prof')
SAVE_DIR  = Path('experiments/online_thesis_v1/preflight/p0_profiling/cprofile_run')
CKPT_DIR  = Path('checkpoints/online_thesis_v1/preflight/p0_profiling/cprofile_run')

# 先清理 prof 文件，避免读到旧数据
if PROF_PATH.exists():
    PROF_PATH.unlink()

# 先生成 manifest（如已存在则跳过）
subprocess.run([
    'python3', '-m', 'scripts.generate_standard_benchmarks',
    '--benchmarks', 'single_u15_upstream_tgt15',
    '--episodes', '30',
], check=True)

t0 = time.time()
subprocess.run([
    'python3', '-m', 'cProfile', '-o', str(PROF_PATH),
    '-m', 'scripts.train_sac',
    '--flow', 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy',
    '--task-geometry', 'upstream',
    '--target-speed', '1.5',
    '--objective', 'efficiency_v2',
    '--probe-layout', 's1',
    '--history-length', '4',
    '--total-steps', '10000',
    '--random-steps', '5000',
    '--update-after', '5000',
    '--batch-size', '256',
    '--hidden-dim', '256',
    '--num-envs', '6',
    '--eval-every', '100000',  # 不 eval（上限大于 total-steps）
    '--checkpoint-every', '100000',
    '--eval-manifest', 'benchmarks/single_u15_upstream_tgt15.json',
    '--seed', '46',
    '--device', 'cuda',
    '--save-dir', str(SAVE_DIR),
    '--checkpoint-dir', str(CKPT_DIR),
], check=True)
elapsed = time.time() - t0
print(f'[P0a] 10k step cProfile run: {elapsed:.1f}s wallclock')
print(f'      prof file size: {PROF_PATH.stat().st_size / 1024:.0f} KB')


[P0a] 10k step cProfile run: 141.2s wallclock
      prof file size: 1500 KB


### P0a 分析：top 30 函数 cumulative time


In [ ]:
import pstats
stats = pstats.Stats(str(PROF_PATH))
stats.strip_dirs().sort_stats('cumulative').print_stats(30)


Sat Apr 25 18:07:43 2026    /tmp/sac_p0a.prof

         36716708 function calls (36352204 primitive calls) in 140.143 seconds

   Ordered by: cumulative time
   List reduced from 9407 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   62.506   62.506 train_utils.py:774(evaluate_agent)
        1    0.000    0.000   62.505   62.505 train_utils.py:273(_evaluate_episodes_serial)
       30    0.061    0.002   62.505    2.083 train_utils.py:216(_rollout_episode)
     3274    0.015    0.000   57.884    0.018 core.py:556(step)
     3274    0.223    0.000   57.822    0.018 env.py:703(step)
    16306    0.306    0.000   53.430    0.003 autopilot.py:231(substep)
    10008    0.033    0.000   44.422    0.004 connection.py:246(recv)
    10008    0.031    0.000   43.914    0.004 connection.py:429(_recv_bytes)
    20016    0.049    0.000   43.869    0.002 connection.py:390(_recv)
    20016   43.797    0.002   43.797    

### P0a 模块分组：env vs torch vs others


In [ ]:
import pstats

stats = pstats.Stats(str(PROF_PATH))
total = stats.total_tt

# 按文件 / 模块分桶
buckets = {
    'auv_nav.vehicle':     0.0,
    'auv_nav.flow':        0.0,
    'auv_nav.env':         0.0,
    'auv_nav.autopilot':   0.0,
    'auv_nav.sac':         0.0,
    'torch':               0.0,
    'numpy':               0.0,
    'gymnasium':           0.0,
    'multiprocessing/IPC': 0.0,
    'other':               0.0,
}
for (file, line, func), (cc, nc, tt, ct, callers) in stats.stats.items():
    f = file or ''
    if 'auv_nav/vehicle' in f:
        buckets['auv_nav.vehicle'] += tt
    elif 'auv_nav/flow' in f:
        buckets['auv_nav.flow'] += tt
    elif 'auv_nav/env' in f:
        buckets['auv_nav.env'] += tt
    elif 'auv_nav/autopilot' in f:
        buckets['auv_nav.autopilot'] += tt
    elif 'auv_nav/sac' in f:
        buckets['auv_nav.sac'] += tt
    elif '/torch/' in f or 'torch/' in f or '_torch' in f:
        buckets['torch'] += tt
    elif '/numpy/' in f or 'numpy/' in f:
        buckets['numpy'] += tt
    elif 'gymnasium' in f:
        buckets['gymnasium'] += tt
    elif 'multiprocessing' in f or 'pickle' in f or 'connection' in f:
        buckets['multiprocessing/IPC'] += tt
    else:
        buckets['other'] += tt

print(f'{"bucket":<25}{"tottime (s)":>14}{"share":>10}')
print('-' * 50)
for k, v in sorted(buckets.items(), key=lambda kv: -kv[1]):
    print(f'{k:<25}{v:>14.2f}{v/total*100:>9.1f}%')
print('-' * 50)
print(f'{"total tottime":<25}{total:>14.2f}{100.0:>9.1f}%')

env_share = (buckets['auv_nav.vehicle'] + buckets['auv_nav.flow']
             + buckets['auv_nav.env']   + buckets['auv_nav.autopilot']) / total
torch_share = buckets['torch'] / total
print()
print(f'env-side total:     {env_share*100:.1f}%')
print(f'torch (NN) total:   {torch_share*100:.1f}%')
print(f'IPC overhead:       {buckets["multiprocessing/IPC"]/total*100:.1f}%')


bucket                      tottime (s)     share
--------------------------------------------------
other                             78.04     55.7%
auv_nav.flow                      20.58     14.7%
numpy                             16.42     11.7%
auv_nav.vehicle                   15.47     11.0%
torch                              5.32      3.8%
gymnasium                          1.24      0.9%
auv_nav.autopilot                  1.23      0.9%
auv_nav.sac                        0.86      0.6%
auv_nav.env                        0.68      0.5%
multiprocessing/IPC                0.31      0.2%
--------------------------------------------------
total tottime                    140.14    100.0%

env-side total:     27.1%
torch (NN) total:   3.8%
IPC overhead:       0.2%


## 3. P0b — num_envs benchmark：6 vs 12

各跑 50k 步 vanilla SAC，记 wallclock，比较 ratio。

判据：
- ratio > 1.7 → 切 `num_envs=12`（thesis matrix 修订）
- 1.4 ≤ ratio ≤ 1.7 → 仍切 12，但提醒 IPC 损耗已接近边界
- ratio < 1.4 → 维持 `num_envs=6`，IPC 已成瓶颈


In [ ]:
import os, subprocess, time
from pathlib import Path

def run_50k(num_envs: int, tag: str) -> float:
    save_dir = Path(f'experiments/online_thesis_v1/preflight/p0_profiling/{tag}')
    ckpt_dir = Path(f'checkpoints/online_thesis_v1/preflight/p0_profiling/{tag}')
    t0 = time.time()
    subprocess.run([
        'python3', '-m', 'scripts.train_sac',
        '--flow', 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy',
        '--task-geometry', 'upstream',
        '--target-speed', '1.5',
        '--objective', 'efficiency_v2',
        '--probe-layout', 's1',
        '--history-length', '4',
        '--total-steps', '50000',
        '--random-steps', '5000',
        '--update-after', '5000',
        '--batch-size', '256',
        '--hidden-dim', '256',
        '--num-envs', str(num_envs),
        '--eval-every', '100000',  # 不 eval
        '--checkpoint-every', '100000',
        '--eval-manifest', 'benchmarks/single_u15_upstream_tgt15.json',
        '--seed', '46',
        '--device', 'cuda',
        '--save-dir', str(save_dir),
        '--checkpoint-dir', str(ckpt_dir),
    ], check=True)
    elapsed = time.time() - t0
    return elapsed

t_n6  = run_50k(num_envs=6,  tag='num_envs_6_50k')
print(f'[P0b] num_envs=6  × 50k: {t_n6:.1f}s = {t_n6/60:.2f} min')

t_n12 = run_50k(num_envs=12, tag='num_envs_12_50k')
print(f'[P0b] num_envs=12 × 50k: {t_n12:.1f}s = {t_n12/60:.2f} min')

ratio = t_n6 / t_n12
print()
print(f'speedup ratio (6 / 12) = {ratio:.2f}x')
print()
if ratio > 1.7:
    decision = '✅ 切 num_envs=12（thesis matrix 修订）'
elif ratio >= 1.4:
    decision = '⚠️  切 num_envs=12，但 IPC 损耗已接近边界'
else:
    decision = '❌ 维持 num_envs=6（IPC 已成瓶颈，并行收益不足）'
print(f'decision: {decision}')


[P0b] num_envs=6  × 50k: 336.8s = 5.61 min
[P0b] num_envs=12 × 50k: 232.7s = 3.88 min

speedup ratio (6 / 12) = 1.45x

decision: ⚠️  切 num_envs=12，但 IPC 损耗已接近边界


## 4. 决策记录 — 落字到 thesis plan §10

把以下两组数字 + 决策落到 [`docs/online_rl_thesis_plan.md`](../docs/online_rl_thesis_plan.md) §10：

```
- P0a 瓶颈分布：env-side _____%，torch _____%，IPC _____%
- P0b num_envs benchmark：6 → ____ s，12 → ____ s，speedup ____x
- num_envs 协议决策：[ ] 维持 6  [ ] 切到 12
- （可选）Numba JIT 待办：env-side > 50% 时打开此选项
```

如果切到 `num_envs=12`，需要：
1. 把 [`docs/online_rl_thesis_plan.md`](../docs/online_rl_thesis_plan.md) §4.1 表格里 `num_envs=6` 改为 `12`
2. 把 [`notebooks/sac_thesis_s2_sensor_envelope.ipynb`](sac_thesis_s2_sensor_envelope.ipynb) 第 9 个 cell 的 `os.environ['NUM_ENVS']` 改为 `'12'`
3. 后续 Sprint 2/4/5 notebook 创建时同步用 `num_envs=12`
